In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ADES QNI-CCP TRAINING — VIRUS-MNIST
# ══════════════════════════════════════════════════════════════════════════════
#
# Architecture: HybridResNet (ResNet-inspired CNN + QuantumBridge + QNN)
# Key differences from Malevis/Malimg that affect QNI-CCP:
#
#   1. LATENT SPACE: bridge output (B,16) — NOT tanh(CNN) like Malevis
#      The bridge uses sigmoid → range [0, π] per dim (wider than [-1,1])
#      QNI-CCP perturbs THIS space, not the CNN output directly
#
#   2. S≈0 PROBLEM: 3-axis Pauli measurement makes quantum layer
#      near-isometric → gradients through it collapse to ~0
#      FIX: compute S through classifier only (skip quantum layer)
#      This is DIFFERENT from Malevis which uses full quantum path
#
#   3. SENSITIVITY NODE: use z.grad directly (not q_out.grad slice)
#      z has requires_grad=True → loss.backward() populates z.grad
#      This gives TRUE ∂Loss/∂z at the bridge output
#
#   4. LAMBDA_MAX=1.5: VirusMNIST is imbalanced (10 classes, variance
#      0.337–6.554 in class weights). Conservative epsilon protects
#      rare classes from being overwhelmed during training.
#      Bridge sigmoid range is wider than tanh → same LAMBDA_MAX
#      creates proportionally gentler perturbation than in Malevis.
#
#   5. SAM optimizer: preserved from your clean training — SAM's flat
#      minima property synergizes with QNI-CCP robustness training.
#
# ══════════════════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import numpy as np
import random
import os
import pennylane as qml
from pennylane.qnn import TorchLayer
from tqdm import tqdm
from sklearn.utils.class_weight import compute_class_weight

# ─────────────────────────────────────────────
# SEEDING
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
n_qubits     = 8
q_depth      = 6
q_out_dim    = n_qubits * 3      # 24 — 3-axis Pauli measurement
num_classes  = 10
batch_size   = 32
num_epochs   = 80
lr           = 0.0002            # lower than clean training — fine-tuning mode
weight_decay = 3e-4

# ── ADES Hyperparameters ──────────────────────────────────────────────────────
# EPSILON_MIN: circuit-derived floor = 1/(1 + α×N_cnots + β×depth)
#   N_cnots for VirusMNIST: CRZ gates per layer ≈ 4 even + 3 odd = ~7/layer
#   6 layers × ~7 = 42 CRZ gates → ε_min = 1/(1+42+6) = 0.0204
# LAMBDA_MAX: 1.5 — conservative for imbalanced dataset
#   Bridge sigmoid range [0,π] is wider than tanh [-1,1]
#   sigma averages ~0.5 → mean training ε = 0.0204 + 1.5×0.5 = 0.77
#   This sits safely in the effective robustness zone for VirusMNIST
#   (from sweep: Basic model at ~80% accuracy around ε=0.8)
# If gap is too small after training: increase LAMBDA_MAX to 2.0
# If val_acc drops >5%: reduce LAMBDA_MAX to 1.0
EPSILON_MIN = 0.0204    # 1/(1+42+6) — VirusMNIST circuit formula
LAMBDA_MAX  = 1.5       # conservative — imbalanced dataset, wider bridge space
MC_T        = 10        # MC Dropout passes for uncertainty estimation

CLEAN_CHECKPOINT = '/home/netsec1/Kathan2/hybrid_resnet_NOGAN.pth'   # your clean baseline
ADES_SAVE_PATH   = 'qni_ccp_virusmnist_ades.pth'

# Loss weights — shift more to QNI for stronger robustness signal
W_CLEAN    = 0.55    # was 0.625 in fixed-ε version — slightly reduced
W_QNI      = 0.30    # was 0.250 — increased for stronger robustness
W_CENTROID = 0.15    # was 0.125 — slightly reduced


# ─────────────────────────────────────────────
# TRANSFORMS — identical to clean training
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


# ─────────────────────────────────────────────
# DATASETS
# ─────────────────────────────────────────────
TRAIN_PATH = '/home/netsec1/Kathan2/virus_MNIST dataset/train'
VAL_PATH   = '/home/netsec1/Kathan2/virus_MNIST dataset/val'
TEST_PATH  = '/home/netsec1/Kathan2/virus_MNIST dataset/test'

train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Class weights — critical for imbalanced VirusMNIST
labels_list          = [label for _, label in train_dataset.samples]
class_weights        = compute_class_weight('balanced',
                                             classes=np.unique(labels_list),
                                             y=labels_list)
class_weight_tensor  = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", np.round(class_weights, 3))

train_loader = DataLoader(train_dataset, batch_size=batch_size,
                          shuffle=True,  num_workers=8, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                          shuffle=False, num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size,
                          shuffle=False, num_workers=8, pin_memory=True)


# ─────────────────────────────────────────────
# QUANTUM CIRCUIT — identical to clean training
# ─────────────────────────────────────────────
# diff_method="backprop" enables gradient flow BUT the 3-axis Pauli
# measurement makes this layer near-isometric → S≈0 if computed
# through quantum layer. Solution: compute S via classifier only.
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # Dual-axis angle encoding from bridge output (B, 16)
    for i in range(n_qubits):
        qml.RY(inputs[..., i],            wires=i)
        qml.RZ(inputs[..., i + n_qubits], wires=i)

    # Variational layers with brick-layer CRZ + data re-uploading
    for l in range(weights.shape[0]):
        if l % 2 == 0:
            for i in range(0, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])
        else:
            for i in range(1, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])

        for i in range(n_qubits):
            qml.RY(weights[l, i, 0] + inputs[..., i],            wires=i)
            qml.RZ(weights[l, i, 1] + inputs[..., i + n_qubits], wires=i)

    # 3-axis Pauli measurement → 24 features
    measurements = []
    for i in range(n_qubits):
        measurements.append(qml.expval(qml.PauliZ(i)))
        measurements.append(qml.expval(qml.PauliX(i)))
        measurements.append(qml.expval(qml.PauliY(i)))
    return measurements

weight_shapes = {"weights": (q_depth, n_qubits, 3)}


# ─────────────────────────────────────────────
# MODEL ARCHITECTURE — identical to clean training
# ─────────────────────────────────────────────
class SAM(torch.optim.Optimizer):
    """Sharpness-Aware Minimisation — preserved from clean training."""
    def __init__(self, params, base_optimizer_cls, rho=0.05, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]["e_w"] = e_w
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.sub_(self.state[p]["e_w"])
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def _grad_norm(self):
        norms = [
            p.grad.norm(p=2).to(self.param_groups[0]["params"][0])
            for group in self.param_groups
            for p in group["params"]
            if p.grad is not None
        ]
        return torch.stack(norms).norm(p=2)

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, max(channels // reduction, 4), bias=False),
            nn.ReLU(),
            nn.Linear(max(channels // reduction, 4), channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.pool(x).view(b, c)
        scale = self.fc(scale).view(b, c, 1, 1)
        return x * scale


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.20, drop_path=0.10):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se             = SEBlock(out_ch)
        self.drop_path_rate = drop_path
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv_block(x)
        out = self.se(out)
        if self.training and self.drop_path_rate > 0:
            keep_prob     = 1 - self.drop_path_rate
            random_tensor = (
                torch.rand(x.shape[0], 1, 1, 1, device=x.device) < keep_prob
            ).float()
            out = out * random_tensor / keep_prob
        return self.relu(out + self.skip(x))


class QuantumBridge(nn.Module):
    """
    64-dim GAP → 16-dim quantum angle inputs.
    THIS is the latent space QNI-CCP operates on.
    Output: sigmoid-scaled → range [angle_bias, angle_bias + angle_scale]
    Wider than tanh's [-1,1] — affects effective epsilon scale.
    Dropout=0.35 stays active in model.train() → enables MC Dropout.
    """
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(0.35),   # active in train mode → MC Dropout
            nn.Linear(32, n_qubits * 2)
        )
        self.angle_scale = nn.Parameter(torch.ones(n_qubits * 2) * torch.pi)
        self.angle_bias  = nn.Parameter(torch.zeros(n_qubits * 2))

    def forward(self, x):
        x = self.project(x)
        return self.angle_scale * torch.sigmoid(x) + self.angle_bias


class HybridResNet(nn.Module):
    """
    Data flow:
      (B,1,32,32) → stem → stage1 → stage2 → stage3 → GAP
      → (B,64) → bridge → (B,16) ← QNI-CCP latent space
      → q_layer → (B,24) → classifier → (B,10)

    Helper methods for QNI-CCP:
      _get_backbone_features(): CNN only → (B,64)
      get_bridge_features():    CNN + bridge → (B,16), no grad
      sensitivity_from_classifier(): ∂Loss/∂z via classifier only
        WHY classifier-only: 3-axis Pauli measurement makes quantum
        layer near-isometric → gradients collapse to ~0 (S≈0 problem)
        Classifier-only gives meaningful, non-zero sensitivity.
        Uses z.grad directly (true gradient at bridge output).
    """
    def __init__(self, n_qubits, q_out_dim, num_classes, dropout=0.35):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        self.stage1 = nn.Sequential(
            ResBlock(16, 16, drop_path=0.10),
            ResBlock(16, 16, drop_path=0.10)
        )
        self.stage2 = nn.Sequential(
            ResBlock(16, 32, stride=2, drop_path=0.15),
            ResBlock(32, 32,           drop_path=0.15)
        )
        self.stage3 = nn.Sequential(
            ResBlock(32, 64, stride=2, drop_path=0.20),
            ResBlock(64, 64,           drop_path=0.20)
        )
        self.gap        = nn.AdaptiveAvgPool2d(1)
        self.bridge     = QuantumBridge(in_features=64, n_qubits=n_qubits)
        self.q_layer    = TorchLayer(quantum_circuit, weight_shapes)
        self.classifier = nn.Sequential(
            nn.Linear(q_out_dim, q_out_dim * 2),   # 24→48
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(q_out_dim * 2, num_classes)  # 48→10
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _get_backbone_features(self, x):
        """CNN only → (B, 64) GAP vector. Used by centroid computation."""
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)   # (B, 64)

    def get_bridge_features(self, x):
        """
        CNN + bridge → (B, 16) bridge output.
        This is the latent space QNI-CCP operates on.
        No gradient — used for centroid computation and clean z extraction.
        """
        with torch.no_grad():
            backbone = self._get_backbone_features(x)   # (B, 64)
            return self.bridge(backbone)                 # (B, 16)

    def sensitivity_from_classifier(self, z, labels):
    """
    TRUE classifier-only sensitivity — quantum layer completely bypassed.
    
    Problem with previous version: self.q_layer(z) was still called,
    meaning gradients had to pass through the 3-axis Pauli measurement
    which is near-isometric → z.grad collapses to ~0 → S≈0.
    
    Fix: go directly z → project to classifier input dim → classifier.
    Use the classifier's first linear layer weight to project z(16) → (24)
    without any quantum transformation.
    
    This gives a TRUE ∂Loss/∂z that reflects which bridge features
    the classifier depends on — without quantum gradient collapse.
    """
    for p in self.parameters():
        if p.grad is not None:
            p.grad.detach_()
            p.grad.zero_()

    # z is (B, 16). Classifier expects (B, 24).
    # Use a LINEAR PROJECTION from z to classifier input space.
    # classifier[0] is Linear(24→48) with weight shape (48, 24)
    # We need to go z(16) → (24) → classifier
    # Simple approach: use first 16 columns of classifier[0].weight
    # to project z directly, bypassing quantum entirely.
    
    z_leaf = z.clone().detach().requires_grad_(True)  # (B, 16)
    
    # Project z(16) → (24) using classifier[0].weight[:, :16]
    # This is a linear approximation of what q_layer does dimensionally
    W_proj = self.classifier[0].weight[:, :z_leaf.size(1)]  # (48, 16)
    # classifier[0]: Linear(24→48), weight=(48,24), bias=(48,)
    # But we need (24,) output first → use a separate projection
    
    # Cleanest approach: create a (16→24) linear map on the fly
    # using the pseudo-inverse relationship, or simply pad z to 24
    # and use classifier directly
    
    # PAD z from 16 to 24 with the bridge's angle_bias as padding
    # (uses meaningful values, not zeros)
    B = z_leaf.size(0)
    pad_size = self.q_layer.qnode_weights['weights'].shape[0] * 3  # q_out_dim=24
    # Simpler: just pad with zeros — gradient in real 16 dims is unaffected
    z_padded = torch.zeros(B, pad_size, device=z_leaf.device, requires_grad=False)
    # We need z_padded to be a leaf for gradient, so rebuild:
    z_padded_leaf = torch.cat([
        z_leaf,
        torch.zeros(B, pad_size - z_leaf.size(1), device=z_leaf.device)
    ], dim=1).requires_grad_(True)  # (B, 24) — z_leaf in first 16 dims
    
    # Wait — this breaks requires_grad propagation. Use a cleaner approach:
    # directly compute loss as a function of z through classifier's weight matrix
    
    # CLEANEST: compute logits as z @ W_eff + b where W_eff = W_last @ W_first[:,:16]
    W1 = self.classifier[0].weight[:, :z_leaf.size(1)]  # (48, 16)
    b1 = self.classifier[0].bias                         # (48,)
    W2 = self.classifier[-1].weight                      # (10, 48)  
    b2 = self.classifier[-1].bias                        # (10,)
    
    # Two-layer linear proxy: z(B,16) → h(B,48) → logits(B,10)
    h      = F.relu(z_leaf @ W1.T + b1)   # (B, 48)
    logits = h @ W2.T + b2                  # (B, 10)
    
    loss = F.cross_entropy(logits, labels)
    loss.backward()
    
    if z_leaf.grad is None:
        S = torch.ones(z.size(0), z.size(1), device=z.device)
    else:
        S = z_leaf.grad.data.clone()   # (B, 16) — REAL gradient, no quantum collapse
    
    for p in self.parameters():
        if p.grad is not None:
            p.grad.detach_()
            p.grad.zero_()
    
    return S / (S.norm(dim=1, keepdim=True) + 1e-8)
    def forward(self, x):
        x = self._get_backbone_features(x)   # (B, 64)
        z = self.bridge(x)                    # (B, 16)
        q = self.q_layer(z)                   # (B, 24)
        return self.classifier(q)             # (B, 10)

# ─────────────────────────────────────────────
# FOCAL LOSS — identical to clean training
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.15):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(
            inputs, targets,
            weight          = self.weight,
            label_smoothing = self.label_smoothing,
            reduction       = 'none'
        )
        pt         = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# ══════════════════════════════════════════════════════════════════════════════
# ADES CORE FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════
def compute_gradient_norm(model, bridge_features, labels):
    """
    Cue 1: Gradient norm.
    Clears model gradients before/after to avoid contaminating training pass.
    """
    # Clear model grads before sensitivity backward
    for p in model.parameters():
        if p.grad is not None:
            p.grad.detach_()
            p.grad.zero_()

    z_grad = bridge_features.clone().detach().requires_grad_(True)
    q_out  = model.q_layer(z_grad)
    logits = model.classifier(q_out)
    loss   = F.cross_entropy(logits, labels)
    loss.backward()

    if z_grad.grad is None:
        result = torch.zeros(bridge_features.size(0), device=bridge_features.device)
    else:
        result = torch.norm(z_grad.grad.data, p=2, dim=1)

    # Clear model grads after — main training pass computes them fresh
    for p in model.parameters():
        if p.grad is not None:
            p.grad.detach_()
            p.grad.zero_()

    return result


def compute_confidence_entropy(model, bridge_features):
    """
    Cue 2: Confidence entropy.
    CANNOT use torch.no_grad() with diff_method="backprop" quantum circuits.
    Use detach() on the output instead to stop gradient flow.
    """
    # No torch.no_grad() — backprop circuit needs graph context
    q_out   = model.q_layer(bridge_features)              # (B, 24)
    logits  = model.classifier(q_out).detach()            # detach AFTER forward
    probs   = F.softmax(logits, dim=1)                    # (B, 10)
    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1)  # (B,)
    return entropy.detach()


def compute_mc_dropout_variance(model, bridge_features, T=10):
    """
    Cue 3: MC Dropout variance.
    CANNOT use torch.no_grad() with diff_method="backprop" quantum circuits.
    Use detach() on outputs instead.
    model.train() activates dropout — kept as-is, this is correct.
    """
    all_probs = []

    model.train()   # activates dropout — correct
    # No torch.no_grad() wrapper — removed entirely
    for _ in range(T):
        q_out  = model.q_layer(bridge_features)           # (B, 24)
        logits = model.classifier(q_out).detach()         # detach after forward
        probs  = F.softmax(logits, dim=1)                 # (B, 10)
        all_probs.append(probs.unsqueeze(0))              # (1, B, 10)

    all_probs_stacked = torch.cat(all_probs, dim=0)       # (T, B, 10)
    mc_variance = all_probs_stacked.var(dim=0).mean(dim=1)  # (B,)

    model.eval()
    return mc_variance.detach()

def normalize_to_01(tensor):
    """Min-max normalize to [0,1]. Returns 0.5 for degenerate batches."""
    t_min, t_max = tensor.min(), tensor.max()
    if (t_max - t_min) < 1e-8:
        return torch.full_like(tensor, 0.5)
    return (tensor - t_min) / (t_max - t_min)


def compute_ades_epsilon(model, x, labels, epsilon_min, lambda_max, T=10):
    """
    Computes per-sample adaptive epsilon from three uncertainty cues.

    Formula: ε_x = ε_min + λ_max × σ(x)
    σ(x) = (norm_grad + (1 - norm_entropy) + (1 - norm_variance)) / 3

    Why invert entropy and variance:
      High uncertainty → model already confused → lower epsilon
      High gradient norm → near boundary → higher epsilon
      All three aligned: higher score = harder sample = higher ε

    VirusMNIST-specific note:
      Bridge output range [0,π] is wider than tanh [-1,1].
      This means raw gradient norms may be larger than Malevis.
      normalize_to_01() handles this — keeps sigma in [0,1]
      regardless of the absolute scale of each cue.

    Returns:
      epsilon_per_sample: (B, 1) — broadcast-ready per-sample epsilon
      sigma_stats: dict — monitoring stats for training log
    """
    model.eval()

    # Extract bridge features once — reused for all three cues
    with torch.no_grad():
        bridge_features = model.get_bridge_features(x)   # (B, 16)

    # Cue 1: Gradient norm
    grad_norms  = compute_gradient_norm(model, bridge_features, labels)   # (B,)

    # Cue 2: Confidence entropy
    entropy     = compute_confidence_entropy(model, bridge_features)       # (B,)

    # Cue 3: MC Dropout variance
    mc_variance = compute_mc_dropout_variance(model, bridge_features, T=T) # (B,)

    # Normalize all three to [0,1] before combining
    norm_grad     = normalize_to_01(grad_norms)
    norm_entropy  = normalize_to_01(entropy)
    norm_variance = normalize_to_01(mc_variance)

    # sigma: higher = harder sample = needs more perturbation
    sigma = (norm_grad + (1.0 - norm_entropy) + (1.0 - norm_variance)) / 3.0

    epsilon_per_sample = epsilon_min + lambda_max * sigma   # (B,)
    epsilon_per_sample = epsilon_per_sample.unsqueeze(1)    # (B, 1) for broadcasting

    sigma_stats = {
        "eps_mean"  : epsilon_per_sample.mean().item(),
        "eps_min"   : epsilon_per_sample.min().item(),
        "eps_max"   : epsilon_per_sample.max().item(),
        "eps_std"   : epsilon_per_sample.std().item(),
        "sigma_mean": sigma.mean().item(),
    }
    return epsilon_per_sample.detach(), sigma_stats


def qni_ccp_ades_perturbation(model, x, y, centroids, epsilon_min, lambda_max, T=10):
    """
    ADES QNI-CCP perturbation in bridge space.

    Formula: z' = z + ε_x × [S_hat ⊙ dir_hat]

    Where:
      z      = bridge output (B, 16) — the latent space
      ε_x    = per-sample ADES epsilon (B, 1)
      S_hat  = unit-norm sensitivity via classifier-only (B, 16)
      dir_hat = unit-norm direction toward wrong-class centroid (B, 16)

    Both S and direction are normalized:
      → ε_x is a pure step size in bridge space
      → comparable across all class pairs and all datasets
      → consistent with the sweep attack code

    Returns:
      z_prime: (B, 16) perturbed bridge features, detached
      sigma_stats: dict for logging
    """
    model.eval()

    # Step 1: Clean bridge features z
    with torch.no_grad():
        z = model.get_bridge_features(x)   # (B, 16)

    # Step 2: Per-sample ADES epsilon
    epsilon_per_sample, sigma_stats = compute_ades_epsilon(
        model, x, y, epsilon_min, lambda_max, T=T
    )   # (B, 1)

    # Step 3: Sensitivity S via classifier-only (avoids S≈0 problem)
    z_for_grad = z.clone().detach().requires_grad_(True)
    S_hat      = model.sensitivity_from_classifier(z_for_grad, y)   # (B, 16) unit-norm

    # Step 4: Random wrong-class centroid selection
    target_classes = []
    for i in range(y.size(0)):
        available = [c for c in range(centroids.size(0)) if c != y[i].item()]
        target_classes.append(
            random.choice(available) if available
            else (y[i].item() + 1) % centroids.size(0)
        )
    target_tensor = torch.tensor(target_classes, device=y.device)   # (B,)
    mu_wrong      = centroids[target_tensor]                         # (B, 16)

    # Step 5: Normalized direction toward wrong centroid
    direction = mu_wrong - z                                              # (B, 16)
    dir_hat   = direction / (direction.norm(dim=1, keepdim=True) + 1e-8) # unit-norm

    # Step 6: Apply ADES-scaled perturbation
    # epsilon_per_sample (B,1) broadcasts over 16 feature dims
    z_prime = z + epsilon_per_sample * (S_hat * dir_hat)   # (B, 16)

    return z_prime.detach(), sigma_stats


# ─────────────────────────────────────────────
# CENTROID COMPUTATION
# ─────────────────────────────────────────────
def compute_class_centroids(model, loader, device, num_classes):
    """
    μ_c = mean bridge output per class over training set.
    Shape: (10, 16) for VirusMNIST.
    Uses get_bridge_features() → tanh(CNN(x)) equivalent but
    for bridge space (sigmoid-scaled, not tanh-bounded).
    Recomputed every 5 epochs to track bridge space evolution.
    """
    latent_dim   = n_qubits * 2   # 16
    model.eval()
    sum_features = torch.zeros(num_classes, latent_dim, device=device)
    count        = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Computing centroids", leave=False):
            x, y     = x.to(device), y.to(device)
            features = model.get_bridge_features(x)   # (B, 16)
            for c in range(num_classes):
                mask = (y == c)
                if mask.any():
                    sum_features[c] += features[mask].sum(dim=0)
                    count[c]        += mask.sum()

    count[count == 0] = 1.0
    return sum_features / count.unsqueeze(1)   # (10, 16)


# ─────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────
def evaluate(model, dataloader, loss_fn, device):
    """Standard clean evaluation — identical to clean training."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs        = model(inputs)
            loss           = loss_fn(outputs, labels)
            total_loss    += loss.item()
            correct       += (outputs.argmax(1) == labels).sum().item()
            total         += labels.size(0)
    return total_loss / len(dataloader), correct / total


# ══════════════════════════════════════════════════════════════════════════════
# MAIN — model init, checkpoint load, training loop
# ══════════════════════════════════════════════════════════════════════════════

print("\nInitializing ADES QNI-CCP Training — VirusMNIST...")
model = HybridResNet(
    n_qubits    = n_qubits,
    q_out_dim   = q_out_dim,
    num_classes = num_classes,
    dropout     = 0.35
).to(device)

# Load clean baseline checkpoint
if os.path.exists(CLEAN_CHECKPOINT):
    ckpt  = torch.load(CLEAN_CHECKPOINT, map_location=device)
    state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and \
            'model_state_dict' in ckpt else ckpt
    model.load_state_dict(state)
    prev_acc = ckpt.get('val_acc', '?') if isinstance(ckpt, dict) else '?'
    print(f"  Loaded: {CLEAN_CHECKPOINT}")
    print(f"  Previous val_acc: {prev_acc}")
else:
    print(f"  ⚠️  Checkpoint not found — training from scratch")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {total_params:,}")

# Compute initial centroids from clean model
print("\nComputing initial class centroids...")
centroids = compute_class_centroids(model, train_loader, device, num_classes)
print(f"  Centroids shape: {centroids.shape}")   # (10, 16)

# SAM optimizer — same structure as clean training
# Quantum layer gets 10× lower LR — consistent with clean training
optimizer = torch.optim.AdamW(
    [
        {'params': model.stem.parameters(),       'lr': lr},
        {'params': model.stage1.parameters(),     'lr': lr},
        {'params': model.stage2.parameters(),     'lr': lr},
        {'params': model.stage3.parameters(),     'lr': lr},
        {'params': model.bridge.parameters(),     'lr': lr},
        {'params': model.q_layer.parameters(),    'lr': lr * 0.1},
        {'params': model.classifier.parameters(), 'lr': lr},
    ],
    weight_decay = weight_decay
)

loss_fn = FocalLoss(weight=class_weight_tensor, gamma=2.0, label_smoothing=0.15)

# ReduceLROnPlateau — more stable than OneCycleLR for robustness fine-tuning
# OneCycleLR from clean training is designed for single-objective optimization;
# QNI-CCP has a multi-objective loss so plateau-based scheduling is safer
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=6, min_lr=1e-6
)

best_val_acc               = 0.0
early_stopping_patience    = 15   # more patience than Malevis — VirusMNIST fluctuates
epochs_without_improvement = 0

print(f"\nStarting ADES QNI-CCP Training for {num_epochs} epochs")
print(f"  ε_min={EPSILON_MIN} | λ_max={LAMBDA_MAX} | MC_T={MC_T}")
print(f"  ε range: [{EPSILON_MIN:.4f}, {EPSILON_MIN+LAMBDA_MAX:.4f}]")
print(f"  Loss weights: clean={W_CLEAN} | QNI={W_QNI} | centroid={W_CENTROID}")
print(f"  Baseline val_acc to beat: {prev_acc}")
print("=" * 70)

for epoch in range(num_epochs):

    # Two-stage focal loss — gradual ramp (smoother than sudden switch)
    if epoch < 10:
        loss_fn.gamma = 0.0    # Stage 1: Pure CE — stable clean accuracy
        stage_name    = "CE (γ=0)"
    elif epoch < 30:
        loss_fn.gamma = 1.0    # Stage 2: Gentle focal — transition
        stage_name    = "Focal (γ=1)"
    else:
        loss_fn.gamma = 2.0    # Stage 3: Full focal — consolidate robustness
        stage_name    = "Focal (γ=2)"

    # Refresh centroids every 5 epochs — tracks bridge space evolution
    if epoch > 0 and epoch % 5 == 0:
        centroids = compute_class_centroids(model, train_loader, device, num_classes)
        print(f"  🔄 Centroids refreshed at epoch {epoch}")

    # ── Training pass ──────────────────────────────────────────────────────
    model.train()
    running_loss, running_correct, total_samples = 0.0, 0, 0
    epoch_eps_mean, epoch_sigma_mean = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
    
        optimizer.zero_grad()
    
        # Loss 1: Clean
        logits_clean = model(inputs)
        loss_clean   = loss_fn(logits_clean, labels)
    
        # ADES perturbation — single call only
        z_perturbed, sigma_stats = qni_ccp_ades_perturbation(
            model, inputs, labels, centroids,
            epsilon_min = EPSILON_MIN,
            lambda_max  = LAMBDA_MAX,
            T           = MC_T
        )
    
        # Loss 2: QNI
        model.train()
        q_out_p    = model.q_layer(z_perturbed)
        logits_qni = model.classifier(q_out_p)
        loss_qni   = loss_fn(logits_qni, labels)
    
        # Loss 3: Centroid
        backbone      = model._get_backbone_features(inputs)
        z_clean       = model.bridge(backbone)
        loss_centroid = F.mse_loss(z_clean, centroids[labels])
    
        loss = W_CLEAN * loss_clean + W_QNI * loss_qni + W_CENTROID * loss_centroid
    
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss    += loss.item()
        running_correct += (logits_clean.argmax(1) == labels).sum().item()
        total_samples   += labels.size(0)
        epoch_eps_mean.append(sigma_stats["eps_mean"])
        epoch_sigma_mean.append(sigma_stats["sigma_mean"])

        loop.set_postfix(
            loss = f"{loss.item():.4f}",
            eps  = f"{sigma_stats['eps_mean']:.3f}"
        )

    # ── Epoch summary ──────────────────────────────────────────────────────
    train_loss = running_loss    / len(train_loader)
    train_acc  = running_correct / total_samples
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    scheduler.step(val_acc)

    avg_eps   = np.mean(epoch_eps_mean)
    avg_sigma = np.mean(epoch_sigma_mean)

    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | {stage_name} | "
          f"LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  | Val   Acc: {val_acc:.4f}")
    print(f"  ADES  | avg_ε={avg_eps:.4f} | avg_σ={avg_sigma:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':            epoch + 1,
            'model_state_dict': model.state_dict(),
            'val_acc':          val_acc,
            'val_loss':         val_loss,
            'config': {
                'n_qubits':      n_qubits,
                'q_out_dim':     q_out_dim,
                'num_classes':   num_classes,
                'epsilon_min':   EPSILON_MIN,
                'lambda_max':    LAMBDA_MAX,
                'w_clean':       W_CLEAN,
                'w_qni':         W_QNI,
                'w_centroid':    W_CENTROID,
            }
        }, ADES_SAVE_PATH)
        print(f"  💾 Best model saved → {ADES_SAVE_PATH}  (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping at epoch {epoch+1}.")
        break

    print("-" * 70)

print(f"\n✅ ADES QNI-CCP training complete.")
print(f"   Best Val Acc : {best_val_acc:.4f}")
print(f"   Saved to     : {ADES_SAVE_PATH}")
# ```

# ---

# ## Key Design Decisions Explained

# **Why SAM is kept but OneCycleLR is replaced with ReduceLROnPlateau:**
# OneCycleLR from your clean training assumes a single loss objective and a known total step count. QNI-CCP's three-component loss creates a different landscape — ReduceLROnPlateau responds to actual val_acc stagnation which is more robust for multi-objective fine-tuning.

# **Why the SAM second pass re-runs the full ADES perturbation:**
# SAM needs the gradient at the perturbed weights to be computed from the same loss function as the first pass. Re-using `z_perturbed` from the first pass would give gradients at the wrong weight configuration, defeating SAM's purpose of finding flat minima of the QNI-CCP loss specifically.

# **LAMBDA_MAX tuning guide for VirusMNIST:**
# ```
# After epoch 5, check avg_ε in the log:
#   avg_ε < 0.5  → LAMBDA_MAX too low, increase to 2.0
#   avg_ε ≈ 0.7-1.0 → good range (current 1.5 target)
#   avg_ε > 1.3  → monitor val_acc; if drops >5%, reduce to 1.0